In [1]:
import pandas as pd
from detoxify import Detoxify
from tqdm import tqdm
import os

# 1. Cargamos el archivo Parquet (igual de fácil que un CSV)
ruta_entrada = "data/sharechat_sample_balanced.parquet" 
ruta_salida = "data/sharechat_toxicidad_evaluada.parquet"

print("Cargando datos...")
df = pd.read_parquet(ruta_entrada)
print(f"Datos cargados: {len(df)} filas.")

# 2. Inicializamos Detoxify (descargará el modelo de ~1GB la primera vez)
print("Cargando modelo multilingüe de toxicidad...")
modelo_tox = Detoxify('multilingual')

# 3. Procesamiento por lotes adaptado para CPU
TAMANO_LOTE = 16 # Lote más pequeño para no ahogar el procesador
resultados_toxicidad = []
textos = df['plain_text'].fillna("").astype(str).tolist()

print(f"Iniciando evaluación de {len(textos)} textos en el CPU...")

# tqdm nos da la barra de progreso y el tiempo estimado
for i in tqdm(range(0, len(textos), TAMANO_LOTE), desc="Evaluando toxicidad"):
    lote = textos[i : i + TAMANO_LOTE]
    
    # Inferencia
    predicciones = modelo_tox.predict(lote)
    df_lote = pd.DataFrame(predicciones)
    resultados_toxicidad.extend(df_lote.to_dict('records'))

# 4. Unimos los resultados matemáticos a nuestros datos originales
df_puntuaciones = pd.DataFrame(resultados_toxicidad)
df_final = pd.concat([df.reset_index(drop=True), df_puntuaciones], axis=1)

# 5. Aplicamos el umbral (Threshold) validado por la literatura (0.1)
df_final['is_toxic'] = df_final['toxicity'] >= 0.1

# 6. Guardamos el resultado enriquecido
df_final.to_parquet(ruta_salida)
print(f"\n¡Proceso completado! Archivo guardado en: {ruta_salida}")

# Ver cuántos de nuestros 20,000 prompts resultaron realmente tóxicos
print("\nResumen de toxicidad encontrada:")
print(df_final['is_toxic'].value_counts())

Cargando datos...
Datos cargados: 14525 filas.
Cargando modelo multilingüe de toxicidad...
Downloading: "https://github.com/unitaryai/detoxify/releases/download/v0.4-alpha/multilingual_debiased-0b549669.ckpt" to C:\Users\t14/.cache\torch\hub\checkpoints\multilingual_debiased-0b549669.ckpt


100%|██████████| 1.04G/1.04G [30:04<00:00, 616kB/s]  


config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

c:\Users\t14\miniconda3\envs\wildchat\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\t14\.cache\huggingface\hub\models--xlm-roberta-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Iniciando evaluación de 14525 textos en el CPU...


Evaluando toxicidad: 100%|██████████| 908/908 [1:41:52<00:00,  6.73s/it]  


¡Proceso completado! Archivo guardado en: data/sharechat_toxicidad_evaluada.parquet

Resumen de toxicidad encontrada:
is_toxic
False    13862
True       663
Name: count, dtype: int64
